In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 3.11 RLC and AC Circuits

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume III — Classical Electrodynamics",
    number="3.11",
    title="RLC and AC Circuits",
    blurb="The driven oscillator returns in electrical disguise: an RLC circuit obeys "
    "exactly the equation of the damped, driven mass-spring system of §1.2 — "
    "same resonance, same Q, same phase lag, with inductance for mass and 1/C for "
    "stiffness.",
    difficulty="intermediate",
    estimate="100–130 min",
)

## Notebook overview

This notebook is a homecoming. The series RLC circuit, a resistor, inductor, and
capacitor driven by an oscillating source, obeys the equation $L\ddot q+R\dot q+q/C=
V(t)$, and that is, term for term, the damped driven oscillator $m\ddot x+b\dot x+kx=
F(t)$ we already solved in
[§1.2](../01-elementary-mechanics/damped-driven-pendulum.ipynb). Inductance plays the
part of mass, resistance
of friction, and the inverse capacitance $1/C$ of the spring constant. Everything that
took work to understand in mechanics, natural frequency, damping regimes, resonance,
the quality factor $Q$, the phase lag, the exchange of energy, transfers wholesale.
**We have solved this circuit before, in mechanics**, and the point of the notebook is
to feel that.

We build the equation from Kirchhoff's voltage law, using the inductor relation from
Faraday's induction ([§3.7](induction.ipynb)) and the capacitor relation from
electrostatics ([§3.2](potential-energy.ipynb)), then
lay it beside the mechanical oscillator and integrate both to watch them coincide to
machine precision. We meet the **complex impedance** $Z=R+i(\omega L-1/\omega C)$, the
bookkeeping trick that turns differential equations into algebra, find the **resonance**
where the reactances cancel and the current peaks, compute the **quality factor** three
equivalent ways, draw the **phasor diagram** that makes the phase relationships visible,
and watch energy slosh between the inductor and capacitor exactly as kinetic and
potential energy trade in a mass on a spring. The generator of
[§3.7](induction.ipynb) finally drives the
circuit it was built for, closing a loop in the volume.

Everything is in **SI units**. The circuit's transient ring-down is genuine temporal
evolution, so exactly one figure is animated, the energy oscillating between $L$ and
$C$; everything else is a still. Circuit diagrams follow the international **IEC 60617**
standard (a rectangle resistor, not the American zigzag), drawn with the shared
`ecp.draw` circuit primitives.

> **How to read the checks.** Each exercise ends with a `validate` call against an
> independent fact: the RLC charge equal to the mechanical displacement, the natural
> frequency $1/\sqrt{LC}$, the impedance equal to $R$ at resonance, the three forms of
> $Q$ agreeing, the average power $\tfrac12 V_0 I_0\cos\varphi$. A ✓ is strong evidence;
> a ✗ is a prompt to *locate the discrepancy*, not a verdict.
>
> **Scope.** A working review, not a full circuits course. See Nolting, *Theoretical
> Physics 3* {cite}`nolting3`; Griffiths, *Introduction to Electrodynamics*
> {cite}`griffiths_em`; and
> [§1.2](../01-elementary-mechanics/damped-driven-pendulum.ipynb) throughout, of which
> this is the electrical
> twin.

## Theory in brief

### The circuit equation from Kirchhoff

Summing the voltage drops around a series loop, across the inductor ($L\,dI/dt$, from
Faraday's law, [§3.7](induction.ipynb)), the resistor ($RI$), and the capacitor
($q/C$, from [§3.2](potential-energy.ipynb)), and
setting them equal to the source gives

```{math}
:label: eq-rlc
L\,\ddot q + R\,\dot q + \frac{q}{C} = V(t), \qquad I=\dot q .
```

### The mechanical correspondence (the spine)

Equation {eq}`eq-rlc` is term-by-term the damped driven oscillator $m\ddot x+b\dot x+
kx=F(t)$ of [§1.2](../01-elementary-mechanics/damped-driven-pendulum.ipynb), under
the dictionary

```{math}
:label: eq-correspondence
L\leftrightarrow m,\quad R\leftrightarrow b,\quad \frac{1}{C}\leftrightarrow k,\quad
q\leftrightarrow x,\quad V\leftrightarrow F .
```

So the natural frequency is $\omega_0=1/\sqrt{LC}$ (the analogue of $\sqrt{k/m}$) and
the damping rate $\gamma=R/2L$ (the analogue of $b/2m$); the under-, critically, and
over-damped regimes carry across unchanged.

### Transient response

With the source off, the homogeneous solution {eq}`eq-rlc` is the **ring-down**: an
underdamped circuit oscillates at $\approx\omega_0$ with an envelope $e^{-\gamma t}$,
the exact electrical image of the mechanical transient of
[§1.2](../01-elementary-mechanics/damped-driven-pendulum.ipynb) and
[§1.6](../01-elementary-mechanics/integrators.ipynb).

### AC steady state and complex impedance

For $V(t)=V_0\cos\omega t$ the steady-state current is most easily found with the
**complex impedance**. Writing every sinusoid as the real part of a complex
exponential $\propto e^{i\omega t}$ turns each time derivative in {eq}`eq-rlc` into a
factor of $i\omega$, so Kirchhoff's sum becomes ordinary algebra with

```{math}
:label: eq-impedance
Z = R + i\Big(\omega L-\frac{1}{\omega C}\Big), \qquad I_0=\frac{V_0}{|Z|},\quad \varphi=\arg Z,
```

built from $Z_R=R$, $Z_L=i\omega L$, $Z_C=1/i\omega C$. The differential equation
becomes algebra.

### Resonance and the quality factor

The current resonates at $\omega_0$, where the reactances cancel ($\omega L=1/\omega C$),
$|Z|$ is minimal and equal to $R$, and the current is maximal and in phase with the
source. The sharpness is the **quality factor**

```{math}
:label: eq-rlc-resonance
Q = \frac{\omega_0 L}{R} = \frac{1}{\omega_0 R C} = \frac{1}{R}\sqrt{\frac{L}{C}},
```

the same $Q$ as the mechanical oscillator, setting both the resonance width
($\approx\omega_0/Q$) and the ring-down rate.

### Phasors and energy

Representing each sinusoid as a rotating vector (a **phasor**), the voltage across $R$
is in phase with the current, $V_L$ leads by $90°$, and $V_C$ lags by $90°$; their
vector sum is the source voltage,

```{math}
:label: eq-phasors
\mathbf V = \mathbf V_R + \mathbf V_L + \mathbf V_C .
```

Energy oscillates between the inductor ($\tfrac12 LI^2$, the "kinetic" store) and the
capacitor ($\tfrac12 q^2/C$, the "potential" store), dissipated by $R$,

```{math}
:label: eq-rlc-energy
E = \tfrac12 L I^2 + \tfrac12\frac{q^2}{C},
```

exactly the kinetic–potential exchange with friction of the mechanical oscillator.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from scipy.integrate import solve_ivp

from ecp import draw, validate
from ecp.animate import show

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
# Running series-RLC values (chosen so Q = (1/R)√(L/C) = √10 ≈ 3.162)
L_H, R_OHM, C_F = 1e-3, 10.0, 1e-6  # inductance (H), resistance (Ω), capacitance (F)
V0 = 5.0  # source amplitude, V
OMEGA0 = 1.0 / np.sqrt(L_H * C_F)  # natural frequency, rad/s


def impedance(omega, R=R_OHM, L=L_H, C=C_F):
    """Complex impedance of a series RLC branch (eq-impedance).

    Z = R + i(ωL − 1/ωC): the single complex number that turns the circuit's
    differential equation into algebra. Its magnitude sets the current
    amplitude V0/|Z| and its argument the current's phase lag.

    Parameters
    ----------
    omega : float or numpy.ndarray
        Angular drive frequency, in rad/s.
    R, L, C : float, optional
        Resistance (Ω), inductance (H), capacitance (F); default the running values.

    Returns
    -------
    complex or numpy.ndarray
        The complex impedance Z(ω), in ohms.
    """
    return R + 1j * (omega * L - 1.0 / (omega * C))


def rlc_rhs(t, y, L, R, C, drive):
    """State derivative of the series RLC circuit (eq-rlc) for `solve_ivp`.

    Recasts L·q'' + R·q' + q/C = V(t) as a first-order system in the state
    [q, I] with I = dq/dt, the circuit form of the damped driven oscillator.

    Parameters
    ----------
    t : float
        Time, in seconds.
    y : array_like
        State ``[q, I]``: charge (C) and current (A).
    L, R, C : float
        Inductance (H), resistance (Ω), capacitance (F).
    drive : callable
        Source voltage ``drive(t)``, in volts.

    Returns
    -------
    list
        The derivative ``[I, (V(t) - R I - q/C)/L]``.
    """
    q, I = y
    return [I, (drive(t) - R * I - q / C) / L]


def osc_rhs(t, y, m, b, k, drive):
    """State derivative of the mechanical damped driven oscillator (§1.2).

    The same second-order form m·x'' + b·x' + kx = F(t) as :func:`rlc_rhs`,
    kept separate only to make the RLC↔mechanics correspondence explicit.

    Parameters
    ----------
    t : float
        Time, in seconds.
    y : array_like
        State ``[x, v]``: displacement and velocity.
    m, b, k : float
        Mass, damping coefficient, spring constant.
    drive : callable
        Forcing ``F(t)``.

    Returns
    -------
    list
        The derivative ``[v, (F(t) - b v - k x)/m]``.
    """
    x, v = y
    return [v, (drive(t) - b * v - k * x) / m]


def quality_factor(R=R_OHM, L=L_H, C=C_F):
    """The three equivalent forms of the resonance quality factor Q (eq-resonance).

    Q = ω0·L/R = 1/(ω0·R·C) = (1/R)·sqrt(L/C) all give the same number, the
    sharpness of resonance and the slowness of the ring-down, identical to the
    mechanical oscillator's Q.

    Parameters
    ----------
    R, L, C : float, optional
        Circuit values; default the running ones.

    Returns
    -------
    tuple of float
        The three forms ``(ω₀L/R, 1/ω₀RC, (1/R)√(L/C))``.
    """
    w0 = 1.0 / np.sqrt(L * C)
    return w0 * L / R, 1.0 / (w0 * R * C), np.sqrt(L / C) / R

## Exercise 1 — The RLC equation and the mechanical mirror

Kirchhoff's voltage law around the series loop sums the drops across the inductor
($L\,dI/dt$), resistor ($RI$), and capacitor ($q/C$) to the source, giving $L\ddot q+
R\dot q+q/C=V(t)$ {eq}`eq-rlc` ({numref}`fig-rlc-circuit`). Set it beside the damped
driven oscillator of [§1.2](../01-elementary-mechanics/damped-driven-pendulum.ipynb)
and the two are the same equation under $L\leftrightarrow m$,
$R\leftrightarrow b$, $1/C\leftrightarrow k$ {eq}`eq-correspondence`. We are not
learning a new problem; we are re-reading an old one.

**This exercise (worked).** With $L=1\,$mH, $R=10\,\Omega$, $C=1\,\mu$F, $V_0=5\,$V and
a drive at $0.7\,\omega_0$:

1. Integrate the RLC charge $q(t)$ with `scipy.integrate.solve_ivp`
   (`DOP853`).
2. Separately integrate the mechanical $x(t)$ with $m=L$, $b=R$, $k=1/C$.

They must be the same function: we are solving
[§1.2](../01-elementary-mechanics/damped-driven-pendulum.ipynb) again, in
electrical clothing.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    q_rlc,
    x_mech,
    "the RLC charge and the mechanical displacement are the same function "
    "(L↔m, R↔b, 1/C↔k)",
    rtol=1e-9,
    atol=1e-13,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 2 — Natural frequency and damping regimes

Switch the source off, charge the capacitor, and release: the circuit rings
down, just as a plucked mass-spring does. The natural frequency is
$\omega_0=1/\sqrt{LC}$ and the damping rate $\gamma=R/2L$, and the comparison of the
two sorts the behaviour into the same three regimes as
[§1.2](../01-elementary-mechanics/damped-driven-pendulum.ipynb): **underdamped**
($\gamma<\omega_0$, oscillatory decay), **critically damped** ($\gamma=\omega_0$,
fastest non-oscillatory return), and **overdamped** ($\gamma>\omega_0$, sluggish
return).

**This exercise (worked).** Confirm $\omega_0=1/\sqrt{LC}$, then integrate the
source-free circuit ($C$ initially charged) for three resistances spanning the regimes
and watch the ring-down change character ({numref}`fig-rlc-regimes`), the electrical
echo of the mechanical damping regimes.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    OMEGA0, 1.0 / np.sqrt(L_H * C_F), "the natural frequency is 1/√(LC)", rtol=1e-9
)
validate.check(
    gamma_under < OMEGA0
    and abs(gamma_crit / OMEGA0 - 1.0) < 1e-12
    and gamma_over > OMEGA0,
    "the damping regimes follow from γ = R/2L versus ω₀: γ_under < ω₀ = γ_crit < γ_over",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 3 — Energy exchange between L and C

In an underdamped ring-down the total energy does not just decay; it **sloshes**.
Energy pours from the capacitor (electric, $\tfrac12 q^2/C$, the "potential" store)
into the inductor (magnetic, $\tfrac12 LI^2$, the "kinetic" store) and back, twice per
cycle, while the resistor bleeds the total away {eq}`eq-rlc-energy`. This is precisely the
kinetic–potential exchange of a damped mass-spring, where energy trades between
$\tfrac12 mv^2$ and $\tfrac12 kx^2$.

**This exercise (worked).** For the underdamped ring-down, compute $E_L(t)$ and
$E_C(t)$ and confirm they exchange (anti-correlated, $180°$ out of phase) while the
total $E_L+E_C$ decreases monotonically through $R$. The animation
({numref}`fig-rlc-energy`) shows the two reservoirs filling and emptying in antiphase,
the total ebbing away.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.check(
    sloshes and monotonic,
    "energy oscillates between L and C (the fraction swings ≈0↔1) and decays through R",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 4 — Complex impedance

Driven at a single frequency, the circuit is handled most cleanly with the **complex
impedance** {eq}`eq-impedance` $Z(\omega)=R+i(\omega L-1/\omega C)$, which packages the
resistor ($R$), inductor ($i\omega L$), and capacitor ($1/i\omega C$) into one complex
number. The current amplitude is $V_0/|Z|$ and its phase lag is $\arg Z$; the
differential equation has become algebra.

**This exercise (worked).** Build $Z(\omega)$ as a complex array over a
frequency sweep (the `impedance` helper) and plot $|Z|$ and $\arg Z$
({numref}`fig-rlc-impedance`). At $\omega_0$ the inductive and capacitive
reactances cancel, leaving $|Z|=R$ exactly: the circuit looks purely
resistive.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    abs(Z_at_res),
    R_OHM,
    "at resonance the impedance is purely resistive, |Z| = R",
    rtol=1e-6,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 5 — Resonance and the quality factor

The current amplitude $I_0(\omega)=V_0/|Z(\omega)|$ peaks where $|Z|$ dips, at
$\omega_0$ {eq}`eq-rlc-resonance`. How sharp that peak is, and how slowly the circuit rings
down, are one number: the **quality factor** $Q$, which can be written three equivalent
ways, $\omega_0 L/R=1/\omega_0 RC=(1/R)\sqrt{L/C}$. A high-$Q$ circuit (small $R$)
resonates sharply and rings for many cycles; a low-$Q$ one is broad and dead. It is the
same $Q$ that governed the mechanical resonance of
[§1.2](../01-elementary-mechanics/damped-driven-pendulum.ipynb).

**This exercise (worked).** Confirm the three forms of $Q$ agree (here $Q=\sqrt{10}
\approx3.162$), show the current resonates at $\omega_0$, and compare the resonance
curves for a sharp (high-$Q$) and a broad (low-$Q$) circuit
({numref}`fig-rlc-resonance`), with the full width at half maximum tracking
$\omega_0/Q$.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    np.array([Q1, Q2, Q3]),
    np.full(3, Q1),
    "the three forms of Q agree (ω₀L/R = 1/ω₀RC = (1/R)√(L/C))",
    rtol=1e-6,
)
validate.close(omega_peak, OMEGA0, "the current resonates at ω₀", rtol=1e-2)

In [ ]:
# (solution hidden on the public site)


## Exercise 6 — Phasors and phase

A phasor turns each oscillating quantity into a rotating vector, and the algebra of
sinusoids into vector addition {eq}`eq-phasors`. Taking the current as reference, the
resistor voltage $V_R=RI_0$ is in phase with it, the inductor voltage $V_L=\omega L I_0$
leads by $90°$, and the capacitor voltage $V_C=I_0/\omega C$ lags by $90°$; their vector
sum is the source voltage. The angle between source and current is the phase lag
$\varphi=\arg Z$ ({numref}`fig-rlc-phasor`).

**This exercise (worked).** Compute the voltage phasors at a frequency below resonance,
draw them, and read the phase: the current **leads** the source below $\omega_0$
(capacitive), **lags** above it (inductive), and is exactly in phase at $\omega_0$.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    phi_res, 0.0, "voltage and current are in phase at resonance (arg Z = 0)", atol=1e-6
)
validate.check(
    phi_lo < 0 < phi_hi,
    "the current leads below ω₀ (capacitive) and lags above ω₀ (inductive)",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 7 — The AC generator drives the circuit

In [§3.7](induction.ipynb) a loop of $N$ turns spun in a field produced the EMF
$\mathcal{E}=NBA\omega\sin
\omega t$. That generator now drives the very circuit it was built for, completing a
thread of the volume: [§3.7](induction.ipynb) *made* the alternating EMF, and §3.11
*uses* it. The
steady-state current it drives is set, as for any sinusoidal source, by the impedance,
with amplitude $\mathcal{E}_0/|Z(\omega)|$.

**This exercise (student).**

1. Build the generator EMF (state $N$, $B$, $A$, $\omega$) and drive the RLC
   circuit with it (`scipy.integrate.solve_ivp`, `DOP853`).
2. Let the transient die away, and confirm the measured steady-state current
   amplitude matches the impedance prediction $\mathcal{E}_0/|Z|$.

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.close(
    I_amp_measured,
    I_amp_predicted,
    "the generator drives a steady-state current set by the impedance, ℰ₀/|Z|",
    rtol=1e-3,
)

## Exercise 8 — Average power and the power factor

Only the resistor consumes energy; the inductor and capacitor store it and hand it
back. So the time-averaged power delivered by the source is $\langle P\rangle=\tfrac12
V_0 I_0\cos\varphi$, where the **power factor** $\cos\varphi$ measures how much of the
current is in phase with the voltage. At resonance $\varphi=0$, the power factor is one,
and the delivery is maximal; far off resonance the current is mostly reactive and little
power flows.

**This exercise (student).** For a stated drive, integrate the instantaneous
power $V(t)\,I(t)$ over a cycle with `numpy.trapezoid` and confirm it equals
$\tfrac12 V_0 I_0\cos\varphi$, and equivalently $\tfrac12 I_0^2 R$, the
power dissipated in the resistor alone.

In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.close(
    P_avg_int,
    P_avg_formula,
    "the average power is ½V₀I₀cosφ, delivered only to the resistor",
    rtol=1e-3,
)

## Exercise 9 — One equation, two worlds

Step back and see what the whole notebook rests on. A mass on a spring with friction
and a charge in a circuit with resistance obey the *same* second-order linear
differential equation. Resonance, damping, the quality factor, the phase lag, the
exchange of energy between two stores: these are not analogies between two phenomena,
they are one phenomenon wearing two costumes. That is why the computational tools built
in mechanics, the integrators of [§0.7](../00-foundations/ode-solvers.ipynb), the
resonance analysis of
[§1.2](../01-elementary-mechanics/damped-driven-pendulum.ipynb), the energy
bookkeeping of [§1.6](../01-elementary-mechanics/integrators.ipynb), transfer to
circuits without a single change. Physics rewards the
reader who notices that two problems are secretly one.

**This exercise.** Make the identity visible one last time: evaluate the
mechanical resonance curve from its closed form
$F_0/\sqrt{(k-m\omega^2)^2+(b\omega)^2}$ and the electrical one from
$V_0/(|Z|\,\omega)$ (through the `impedance` helper — a different code
path), and confirm they are the same curve to numerical precision. The
forward road runs to coupled circuits (the normal modes of
[§2.7](../02-classical-mechanics/small-oscillations.ipynb)), the quantum
LC oscillator (Vol VI), and the relativistic capstone
([§3.12](relativistic-maxwell.ipynb)) that closes the
volume.

In [ ]:
# (solution hidden on the public site)


### Validation 9

In [ ]:
validate.close(
    elec_curve,
    mech_curve,
    "the mechanical and electrical resonance curves coincide (one equation, two worlds)",
    rtol=1e-6,
)

## Notebook summary

- The series RLC equation $L\ddot q+R\dot q+q/C=V(t)$ {eq}`eq-rlc` is **term-for-term**
  the damped driven oscillator of
  [§1.2](../01-elementary-mechanics/damped-driven-pendulum.ipynb)
  ($L\leftrightarrow m$, $R\leftrightarrow b$,
  $1/C\leftrightarrow k$); integrating both gave identical curves (agreeing to $<10^{-14}$).
- **Natural frequency** $\omega_0=1/\sqrt{LC}$ and damping $\gamma=R/2L$ sort the
  ring-down into the under-, critically, and over-damped regimes of
  [§1.2](../01-elementary-mechanics/damped-driven-pendulum.ipynb); energy
  **sloshes** between $L$ ($\tfrac12 LI^2$) and $C$ ($\tfrac12 q^2/C$), decaying through
  $R$ (animated, anti-correlated, total monotonically falling).
- **Complex impedance** $Z=R+i(\omega L-1/\omega C)$ makes the steady state algebra:
  $|Z|=R$ exactly at resonance, the current peaks at $\omega_0$, and the **quality
  factor** in its three forms agrees at $Q=\sqrt{10}\approx3.16$.
- **Phasors** show the current leading below $\omega_0$ (capacitive) and lagging above
  (inductive), in phase at resonance; the **generator** of
  [§3.7](induction.ipynb) drives a steady current
  $\mathcal{E}_0/|Z|$; the **average power** $\tfrac12 V_0 I_0\cos\varphi=\tfrac12 I_0^2R$
  goes only to the resistor; and the mechanical and electrical resonance curves are the
  same curve. One equation, two worlds.

## Outlook

- **Filters and parallel resonance.** A parallel RLC shows anti-resonance; series and
  parallel combinations build the low-, high-, and band-pass filters of all signal
  processing.
- **Transformers and matching.** Mutual inductance ([§3.7](induction.ipynb)) couples
  circuits to step
  voltages and match impedances; pushed further it becomes the transmission line, where
  circuits meet waves.
- **Coupled circuits.** Two coupled LC loops have normal modes exactly like the coupled
  oscillators of [§2.7](../02-classical-mechanics/small-oscillations.ipynb), splitting
  into symmetric and antisymmetric resonances.
- **The quantum LC oscillator (Vol VI).** Quantizing $\tfrac12 LI^2+\tfrac12 q^2/C$ is
  quantizing a harmonic oscillator; the LC circuit is a build-it-yourself quantum
  oscillator, the heart of superconducting qubits.
- **The capstone ([§3.12](relativistic-maxwell.ipynb)).** Relativistic
  electrodynamics closes the volume, showing
  $\mathbf E$ and $\mathbf B$ to be one object seen from different frames.

In [ ]:
from ecp.style import footer

footer()